# Scraping: http://www.bbc.co.uk/news

Πάμε να πάρουμε περιεχόμενο από την αρχική σελίδα του BBC News. Θέλουμε:

* Τίτλους (Headlines)
* Συνόψεις (Summary)
* Υπερσυνδέσεις (link)

## Ξεκινώντας

Θέλουμε να **εισάγουμε τα απαραίτητα libraries**.

In [3]:
import requests
from bs4 import BeautifulSoup

Προχωράμε να **κατεβάσουμε την ιστοσελίδα** και *να την εισάγουμε στο BeautifulSoup*.

In [9]:
response = requests.get('https://www.bbc.com/news')
doc = BeautifulSoup(response.text, 'html.parser')

Συνήθως την σελίδα που αναλύουμε στο Beautiful Soup την ονομάζουμε `soup`, αλλά σήμερα θα κάνουμε τη διαφορά και θα την ονομάσουμε `doc`, για να θυμόμαστε ότι είναι ολόκληρο το *document*.

## Προσπάθεια 1η: Παίρνουμε απευθείας τα tags

Αν κοιτάξουμε την σελίδα, προσπαθούμε με το βελάκι να επιλέξουμε τα headlines, αλλά δεν **γίνεται**. Δεν μπορούμε να επιλέξουμε κανέναν τίτλο! Προφανώς πρόκειται για ολόκληρο το BLOCK του άρθρου ή κάτι τέτοιο?

Ευτυχώς γνωρίζουμε λίγο HTML, οπότε μπορούμε να επιλέξουμε από το δεξί πλαίσιο που λέγεται Elements. Πάμε στο `h3` tag, το οποίο ξέρουμε ότι περιέχει τον τίτλο και το περιεχόμενο του.

Χμ, άρα μήπως να πάρουμε απλά όλα τα `h3` tags?

In [11]:
headlines = doc.find_all('h2')

for headline in headlines:
    print(headline.text)

Trump and King hold private talks after royals welcomed to White House with crowds and flypast
US political violence generates a familiar cycle - this time it's in overdrive
United Arab Emirates to quit oil cartel Opec
Why Sam Altman and his former hero Elon Musk are taking their toxic feud to court
'It's bizarre': Californians grapple with revelation that suspected Trump gunman was neighbour
Watch: Jimmy Kimmel defends 'expectant widow' joke after first lady criticism
Russian superyacht sails through Strait of Hormuz despite blockade
The other life of US soldier accused of betting on Maduro's removal
Suspected gunman aged 89 held in Greece after five wounded
Canada's Carney launches a sovereign wealth fund. What is it?
Ex-actor Nathan Chasing Horse jailed for at least 37 years for sexual assault
Man admits plotting attack on Taylor Swift concert in Vienna
Japan Airlines trials humanoid robots as ground handlers
More to explore
Watch as Sabastian Sawe's parents celebrate marathon recor

Αρκετά απλό, αλλά? ...λείπει το link και το summary.

Ok, οπότε μπορούμε να δοκιμάσουμε να πάρουμε όλα τα `a` tags, αλλά σίγουρα θα πάρουμε μαζί με αυτά που θέλουμε και αρκετά σκουπίδια που θα περιέχουν `a` tags - όπως π.χ. το περιεχόμενο στο footer της σελίδας κ.λπ. Πάμε να δούμε μήπως μέσα στα `a` tags του άρθρου υπάρχει κάποιο ξεχωριστό class?

Αν παρατηρήσουμε καλά θα δούμε ότι όντως είναι `class="gs-c-promo-heading nw-o-link-split__anchor gs-o-faux-block-link__overlay-link gel-pica-bold"`. Αυτό δεν είναι μόνο ένα class, είναι **ΠΟΛΛΑ classes**.

* `gs-c-promo-heading`
* `nw-o-link-split__anchor`
* `gs-o-faux-block-link__overlay-link`
* `gel-pica-bold`

Κάπου εδώ αρχίζουμε τις μαντεψιές. Ας δοκιμάσουμε πρώτα το `gs-c-promo-heading` που φαίνεται αρκετά λογικό να είναι το σωστό!

In [23]:
base_url = "https://www.bbc.com"

for link in doc.select("a[data-testid='internal-link']"):
    title = link.get_text(strip=True)
    href = link.get("href")

    if href and href.startswith("/news") and title:
        print(title)
        print(base_url + href)
        print()

US political violence generates a familiar cycle - this time it's in overdriveIn modern America, it seems violence of this kind has become an ever-present storm that can strike anywhere and at any moment.16 hrs agoUS & Canada
https://www.bbc.com/news/articles/cje4eeyq27lo

United Arab Emirates to quit oil cartel OpecThe UAE's decision, after nearly 60 years of membership, is seen as a potential death knell for the oil cartel.Just nowBusiness
https://www.bbc.com/news/articles/cj4pxwlr52yo

Why Sam Altman and his former hero Elon Musk are taking their toxic feud to courtThe battle between the AI big hitters has largely played out on social media. Now it is coming to the courtroom.48 mins ago
https://www.bbc.com/news/articles/cn8dedv8w8xo

'It's bizarre': Californians grapple with revelation that suspected Trump gunman was neighbourPeople living near the man charged with the attempted assassination of Trump  say they were shocked to see their neighbour on TV.5 hrs agoUS & Canada
https://w

In [26]:
cards = doc.find_all('div', {'class': 'sc-feaf8701-1 dzYnRP'})

for card in cards:
    title = card.find('h2')
    if title:
        print(title.get_text(strip=True))

Trump and King hold private talks after royals welcomed to White House with crowds and flypast
US political violence generates a familiar cycle - this time it's in overdrive
United Arab Emirates to quit oil cartel Opec
Why Sam Altman and his former hero Elon Musk are taking their toxic feud to court
'It's bizarre': Californians grapple with revelation that suspected Trump gunman was neighbour
Watch: Jimmy Kimmel defends 'expectant widow' joke after first lady criticism
Russian superyacht sails through Strait of Hormuz despite blockade
The other life of US soldier accused of betting on Maduro's removal
Suspected gunman aged 89 held in Greece after five wounded
Canada's Carney launches a sovereign wealth fund. What is it?
Ex-actor Nathan Chasing Horse jailed for at least 37 years for sexual assault
Man admits plotting attack on Taylor Swift concert in Vienna
Japan Airlines trials humanoid robots as ground handlers
Musk v Altman: The tech billionaires and former friends now facing off in 

In [24]:
for link in doc.select("a[data-testid='internal-link'] h2"):
    print(link.get_text(strip=True))

US political violence generates a familiar cycle - this time it's in overdrive
United Arab Emirates to quit oil cartel Opec
Why Sam Altman and his former hero Elon Musk are taking their toxic feud to court
'It's bizarre': Californians grapple with revelation that suspected Trump gunman was neighbour
Watch: Jimmy Kimmel defends 'expectant widow' joke after first lady criticism
Russian superyacht sails through Strait of Hormuz despite blockade
The other life of US soldier accused of betting on Maduro's removal
Suspected gunman aged 89 held in Greece after five wounded
Canada's Carney launches a sovereign wealth fund. What is it?
Ex-actor Nathan Chasing Horse jailed for at least 37 years for sexual assault
Man admits plotting attack on Taylor Swift concert in Vienna
Japan Airlines trials humanoid robots as ground handlers
Musk v Altman: The tech billionaires and former friends now facing off in court
Watch as Sabastian Sawe's parents celebrate marathon record
Will Mexico City's airport be

In [20]:
import requests
from bs4 import BeautifulSoup

url = "https://www.bbc.com/news"
headers = {"User-Agent": "Mozilla/5.0"}

response = requests.get(url, headers=headers)
doc = BeautifulSoup(response.text, "html.parser")

for h in doc.select("[data-testid='card-headline']"):
    parent = h.find_parent("a")

    if parent:
        link = parent.get("href")

        if link:
            # κάνουμε full url
            if link.startswith("/"):
                link = "https://www.bbc.com" + link

            print(link)

https://www.bbc.com/news/live/c4g5lly7qg8t
https://www.bbc.com/news/articles/cje4eeyq27lo
https://www.bbc.com/news/articles/cj4pxwlr52yo
https://www.bbc.com/news/articles/cn8dedv8w8xo
https://www.bbc.com/news/articles/c392knk3w2jo
https://www.bbc.com/news/videos/c62d4n5rzr7o
https://www.bbc.com/news/articles/cm2pn8zdxdjo
https://www.bbc.com/news/articles/clyxd5wrr0wo
https://www.bbc.com/news/articles/cyv2pmz2v86o
https://www.bbc.com/news/articles/c98m800r28qo
https://www.bbc.com/news/articles/cx218vkdvpgo
https://www.bbc.com/news/articles/c05d5qgprjzo
https://www.bbc.com/news/articles/cpwp87j1llvo
https://www.bbc.com/news/articles/cn8dedv8w8xo
https://www.bbc.com/news/videos/cn0pl57xlppo
https://www.bbc.com/news/videos/cz7231qx88qo
https://www.bbc.com/news/articles/c1mk2pr39k7o
https://www.bbc.com/news/articles/c70d249rd0go
https://www.bbc.com/news/articles/cd7jpg4w181o
https://www.bbc.com/news/articles/ce9n794g42zo
https://www.bbc.com/news/videos/c62d4n5rzr7o
https://www.bbc.com/news/

Το αποτέλεσμα φαίνεται μια χαρά! Παίρνει το κείμενο του `h3`, γιατί το `h3` βρίσκεται μέσα στο `a` tag, αλλά δεν περιέχει το *link* του URL. Ας ρίξουμε μια ματιά στο `a` tag...

    <a class="gs-c-promo-heading nw-o-link-split__anchor gs-o-faux-block-link__overlay-link gel-pica-bold" href="/news/world-middle-east-39302560">
   
...το URL κρύβεται μέσα στην ιδιότητα `href`. Αν πάρουμε το link, είναι εύκολο να πάρουμε μετά το attribute, απλά χρησιμοποιούμε το`['href']`

In [ ]:
links = doc.find_all('a', { 'class': 'gs-c-promo-heading' })

for link in links:
    print(link.text)
    print(link['href'])

Ωραίο, ε? Απλά τώρα μας μένει ένα τελευταίο πρόβλημα: **δεν έχουμε τα summaries**. Αλλά οκ, μπορούμε να χρησιμοποιήσουμε τον Inspector για να διαλέξουμε ένα...

    <p class="gs-c-promo-summary gel-long-primer gs-u-mt nw-c-promo-summary">Trade and Nato are high on the agenda as the much-anticipated Washington talks begin.</p>

Και πάλι, πρέπει να διαλέξουμε ένα από όλα αυτά, ας δοκιμάσουμε το `gs-c-promo-summary` που φαίνεται καλό.

In [ ]:
summaries = doc.find_all('p', { 'class': 'gs-c-promo-summary' })

for summary in summaries:
    print(summary.text)

Ωραία, αλλά **τώρα έχουμε κολλήσει:** δεν υπάρχει τρόπος να συνδυάσουμε και τα headlines και τα links με τα summaries και ακόμη και αν υπήρχε (ίσως με `zip`), δεν θα ήμασταν σίγουροι ότι είναι σωστό το ματσάρισμα.

Άρα τι κάνουμε τώρα?

## Προσπάθεια 2η: Parent elements

Όταν επιλέγουμε ένα element - ένα link και το κείμενο του, ή μια λίστα με headlines - μας ενδιαφέρει μόνο το element που κοιτάμε. Μερικές φορές όμως, **πρέπει να πάρουμε πολλά διαφορετικά elements την ίδια στιγμή.** Όταν συμβαίνει αυτό πρέπει να δούμε τι κοινό έχουν όλα αυτά μαζί.

Αν κοιτάξουμε το summary, το link και τον τίτλο, ίσως βρούμε κάτι σαν το παρακάτω. **Είναι γιγάντιο, αλλά τι να κάνουμε αυτό θέλουμε.**

	<div class="gs-c-promo nw-c-promo gs-o-faux-block-link gs-u-pb gs-u-pb+@m gs-c-promo--inline gs-c-promo--stacked@m nw-u-w-auto gs-c-promo--flex" data-entityid="container-top-stories#3">
		<div class="gs-c-promo-image gs-u-display-none gs-u-display-inline-block@xs gel-1/2@xs gel-1/1@m">
			<div class="gs-o-media-island">
				<div class="gs-o-responsive-image gs-o-responsive-image--16by9"><img alt="Yemeni police gather round bodies of Somali refugees (17/03/19)" class="qa-lazyload-image lazyautosizes lazyloaded" data-sizes="auto" data-srcset="https://ichef.bbci.co.uk/live-experience/cps/240/cpsprodpb/14F5A/production/_95205858_hi038526789.jpg 240w, https://ichef.bbci.co.uk/live-experience/cps/320/cpsprodpb/14F5A/production/_95205858_hi038526789.jpg 320w, https://ichef.bbci.co.uk/live-experience/cps/480/cpsprodpb/14F5A/production/_95205858_hi038526789.jpg 480w, https://ichef.bbci.co.uk/live-experience/cps/624/cpsprodpb/14F5A/production/_95205858_hi038526789.jpg 624w, https://ichef.bbci.co.uk/live-experience/cps/800/cpsprodpb/14F5A/production/_95205858_hi038526789.jpg 800w" data-widths="[240,320,480,624,800]" sizes="177px" src="data:image/gif;base64,R0lGODlhAQABAIAAAAAAAP///yH5BAEAAAAALAAAAAABAAEAAAIBRAA7" srcset="https://ichef.bbci.co.uk/live-experience/cps/240/cpsprodpb/14F5A/production/_95205858_hi038526789.jpg 240w, https://ichef.bbci.co.uk/live-experience/cps/320/cpsprodpb/14F5A/production/_95205858_hi038526789.jpg 320w, https://ichef.bbci.co.uk/live-experience/cps/480/cpsprodpb/14F5A/production/_95205858_hi038526789.jpg 480w, https://ichef.bbci.co.uk/live-experience/cps/624/cpsprodpb/14F5A/production/_95205858_hi038526789.jpg 624w, https://ichef.bbci.co.uk/live-experience/cps/800/cpsprodpb/14F5A/production/_95205858_hi038526789.jpg 800w"></div>
			</div>
		</div>
		<div class="gs-c-promo-body gel-1/2@xs gel-1/1@m gs-u-mt@m">
			<div>
				<a class="gs-c-promo-heading nw-o-link-split__anchor gs-o-faux-block-link__overlay-link gel-pica-bold" href="/news/world-middle-east-39302560">
				<h3 class="gs-c-promo-heading__title gel-pica-bold nw-o-link-split__text">Attack on Yemen migrant boat kills 42</h3></a>
				<p class="gs-c-promo-summary gel-long-primer gs-u-mt nw-c-promo-summary">It is unclear who was behind a helicopter attack which killed 42 refugees and injured 80.</p>
			</div>
			<ul class="gs-o-list-inline gs-o-list-inline--divided gel-brevier gs-u-mt-">
				<li><span class="gs-c-timestamp gs-o-bullet gs-o-bullet- nw-c-timestamp"><span class="gs-o-bullet__icon gel-icon"><svg viewbox="0 0 32 32">
				<polygon points="17,15.4 17,6 15,6 15,16.6 23.8,21.7 24.8,19.9"></polygon>
				<path d="M16,4c6.6,0,12,5.4,12,12c0,6.6-5.4,12-12,12S4,22.6,4,16C4,9.4,9.4,4,16,4 M16,0C7.2,0,0,7.2,0,16c0,8.8,7.2,16,16,16 s16-7.2,16-16C32,7.2,24.8,0,16,0L16,0z"></path></svg></span><time class="gs-o-bullet__text date qa-status-date relative-time" data-datetime="1h" data-seconds="1489768430" data-timestamp-inserted="true" datetime="2017-03-17T16:33:50.000Z">48 minutes ago</time></span></li>
				<li>
					<a aria-label="From Middle East" class="gs-c-section-link gs-c-section-link--truncate nw-c-section-link nw-o-link nw-o-link--no-visited-state" href="/news/world/middle_east"><span aria-hidden="true">Middle East</span></a>
				</li>
			</ul>
		</div>
	</div>

To πάνω πάνω κομμάτι λέγεται **parent element**, όλα τα άλλα elements βρίσκονται μέσα σε αυτό. Αν θέλουμε να το πάρουμε όλο μαζί, τότε πρέπει πρώτα να παίρνουμε τον κάθε γονέα (parent) κάθε *είδηση* και μετά να πάρουμε τα επιμέρους κομμάτια που υπάρχουν μέσα π.χ. headline, links, εικόνες κ.λπ.

Tο κομμάτι του class είναι `class="gs-c-promo nw-c-promo gs-o-faux-block-link gs-u-pb gs-u-pb+@m gs-c-promo--inline gs-c-promo--stacked@m nw-u-w-auto gs-c-promo--flex" data-entityid="container-top-stories#3"`, το οποίο εντάξει όντως είναι κάπως τρομακτικό, αλλά θα δοκιμάσουμε το `gs-c-promo` και ίσως δουλέψει.

In [ ]:
stories = doc.find_all('div', { 'class': 'gs-c-promo' })
for story in stories:
    print(story.text)



Προφανώς δεν μπορούμε να χρησιμοποιήσουμε το `.text` γιατί θα πάρει *τα πάντα* που περιέχουν κείμενο, και το headline *και* το summary. Αντιθέτως αυτό που πρέπει να κάνουμε είναι

* ΒΗΜΑ ΕΝΑ: Χρησιμοποιούμε το doc για να πάρουμε την είδηση
* ΒΗΜΑ ΔΥΟ: Χρησιμοποιούμε την είδηση για να πάρουμε τον τίτλο
* ΒΗΜΑ ΤΡΙΑ: Χρησιμοποιούμε την είδηση για να πάρουμε τον link
* ΒΗΜΑ ΤΕΣΣΕΡΑ: Χρησιμοποιούμε την είδηση για να πάρουμε την σύνοψη

### ΒΗΜΑ ΕΝΑ: Χρησιμοποιούμε το doc για να πάρουμε την είδηση (story)

In [ ]:
stories = doc.find_all('div', { 'class': 'gs-c-promo' })
for story in stories:
    print("This is a story")

## ΒΗΜΑ ΔΥΟ + ΤΡΙΑ: Χρησιμοποιούμε την είδηση για να πάρουμε τον τίτλο

Τώρα μπορούμε να κάνουμε ακριβώς το ίδιο για να πάρουμε τον τίτλο και το link, και μετά με το `['href']` να πάρουμε το link του URL.

In [31]:
cards = doc.select(
    'div[data-testid="card-text-wrapper"], '
    'div[data-testid="chester-article"]'
)

for card in cards:
    title = card.select_one('h2[data-testid="card-headline"]')
    desc = card.select_one('p[data-testid="card-description"]')
    link = card.find_parent('a')

    if not title:
        continue

    print("THIS IS A STORY")
    print("Τίτλος:", title.get_text(strip=True))

    if desc:
        print("Περίληψη:", desc.get_text(strip=True))
    else:
        print("Περίληψη: Δεν υπάρχει")

    if link and link.has_attr('href'):
        href = link['href']
        if href.startswith('/'):
            href = 'https://www.bbc.com' + href
        print("Link:", href)
    else:
        print("Link: Δεν βρέθηκε")

    print("-" * 50)

THIS IS A STORY
Τίτλος: US political violence generates a familiar cycle - this time it's in overdrive
Περίληψη: In modern America, it seems violence of this kind has become an ever-present storm that can strike anywhere and at any moment.
Link: https://www.bbc.com/news/articles/cje4eeyq27lo
--------------------------------------------------
THIS IS A STORY
Τίτλος: United Arab Emirates to quit oil cartel Opec
Περίληψη: The UAE's decision, after nearly 60 years of membership, is seen as a potential death knell for the oil cartel.
Link: https://www.bbc.com/news/articles/cj4pxwlr52yo
--------------------------------------------------
THIS IS A STORY
Τίτλος: Why Sam Altman and his former hero Elon Musk are taking their toxic feud to court
Περίληψη: The battle between the AI big hitters has largely played out on social media. Now it is coming to the courtroom.
Link: https://www.bbc.com/news/articles/cn8dedv8w8xo
--------------------------------------------------
THIS IS A STORY
Τίτλος: 'It'

In [35]:
import pandas as pd

data = []

cards = doc.select(
    'div[data-testid="card-text-wrapper"], '
    'div[data-testid="chester-article"]'
)

for card in cards:
    title = card.select_one('h2[data-testid="card-headline"]')
    desc = card.select_one('p[data-testid="card-description"]')
    link = card.find_parent('a')

    if not title:
        continue

    title_text = title.get_text(strip=True)

    desc_text = desc.get_text(strip=True) if desc else "Δεν υπάρχει"

    if link and link.has_attr('href'):
        href = link['href']
        if href.startswith('/'):
            href = 'https://www.bbc.com' + href
    else:
        href = "Δεν βρέθηκε"

    # βάζουμε σε dict
    data.append({
        "Τίτλος": title_text,
        "Περίληψη": desc_text,
        "Link": href
    })

# δημιουργία dataframe
df = pd.DataFrame(data)
# save σε CSV
df.to_csv("bbc_news.csv", index=False, encoding="utf-8-sig")

In [40]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import time

BASE_URL = "https://www.capital.gr"
URL = "https://www.capital.gr/eidiseis?page={}"

headers = {
    "User-Agent": "Mozilla/5.0"
}

data = []

for page in range(1, 10):
    print("Scraping page:", page)

    response = requests.get(URL.format(page), headers=headers, timeout=15)
    response.encoding = "utf-8"

    soup = BeautifulSoup(response.text, "html.parser")

    articles = soup.select("article, div[class*='article'], div[class*='news'], li")

    for article in articles:
        title_tag = article.find(["h2", "h3"])

        if not title_tag:
            continue

        title = title_tag.get_text(strip=True)

        if not title:
            continue

        a = title_tag.find_parent("a") or article.find("a")
        link = ""

        if a and a.has_attr("href"):
            link = urljoin(BASE_URL, a["href"])

        desc_tag = article.find("p")
        description = desc_tag.get_text(strip=True) if desc_tag else ""

        data.append({
            "page": page,
            "title": title,
            "description": description,
            "link": link
        })

    time.sleep(0.5)

df = pd.DataFrame(data)

df = df.drop_duplicates(subset=["title", "link"])

df.to_csv("capital_news_200_pages.csv", index=False, encoding="utf-8-sig")

print("Τέλος")
print("Σύνολο ειδήσεων:", len(df))
print(df.head())

Scraping page: 1
Scraping page: 2
Scraping page: 3
Scraping page: 4
Scraping page: 5
Scraping page: 6
Scraping page: 7
Scraping page: 8
Scraping page: 9
Τέλος
Σύνολο ειδήσεων: 162
   page                                              title  \
0     1  Συνάντηση Κ. Τασούλα με τον Κρις Ράιτ - "Η Ελλ...   
2     1  Οι επενδύσεις διατηρούν τη δυναμική και μετά τ...   
3     1  Συνάντηση Τραμπ με τον βασιλιά Κάρολο - "Οι ΗΠ...   
4     1  Σ. Ζαχαράκη: Η Ελλάδα ενσωματώνει με ταχύτητα ...   
5     1  ΗΠΑ: Οι θεωρίες συνωμοσίας που λένε ότι ο Τραμ...   

                                         description  \
0  Πιο συγκεκριμένα, συζήτησαν για τη συνεργασία ...   
2  Ομαλή μετάβαση από το ΤΑΑ στα νέα χρηματοδοτικ...   
3   "Είναι ένα τεράστιο προνόμιο να σας καλωσορίζω".   
4                      Επαφές στην Κωνσταντινούπολη.   
5  Η τελευταία θεωρία λέει ότι ο Τραμπ είναι "μετ...   

                                                link  
0  https://www.capital.gr/politiki/3988249/sunant...  


In [41]:
pd.DataFrame(data)

,page,title,description,link
0,1,"Συνάντηση Κ. Τασούλα με τον Κρις Ράιτ - ""Η Ελλ...","Πιο συγκεκριμένα, συζήτησαν για τη συνεργασία ...",https://www.capital.gr/politiki/3988249/sunant...
1,1,"Συνάντηση Κ. Τασούλα με τον Κρις Ράιτ - ""Η Ελλ...","Πιο συγκεκριμένα, συζήτησαν για τη συνεργασία ...",https://www.capital.gr/politiki/3988249/sunant...
2,1,Οι επενδύσεις διατηρούν τη δυναμική και μετά τ...,Ομαλή μετάβαση από το ΤΑΑ στα νέα χρηματοδοτικ...,https://www.capital.gr/oikonomia/3988206/oi-ep...
3,1,"Συνάντηση Τραμπ με τον βασιλιά Κάρολο - ""Οι ΗΠ...","""Είναι ένα τεράστιο προνόμιο να σας καλωσορίζω"".",https://www.capital.gr/diethni/3988247/sunanti...
4,1,Σ. Ζαχαράκη: Η Ελλάδα ενσωματώνει με ταχύτητα ...,Επαφές στην Κωνσταντινούπολη.,https://www.capital.gr/politiki/3988244/s-zaxa...
...,...,...,...,...
175,9,Unicef: Το Αφγανιστάν κινδυνεύει να χάσει ως κ...,Οι Ταλιμπάν έχουν περιορίσει σημαντικά τον αρι...,https://www.capital.gr/diethni/3988005/unicef-...
176,9,Επιχείρηση ανάταξης της κυβερνητικής φθοράς,"""Όχημα"" η οικονομία. Μηνύματα Μητσοτάκη σε αγο...",https://www.capital.gr/politiki/3988004/epixei...
177,9,Νέο άλμα για το πετρέλαιο καθώς οι αγορές περι...,Άλμα κοντά στο 3% σημειώνουν οι τιμές του αργο...,https://www.capital.gr/agores/3988003/neo-alma...
178,9,Ιαπωνία: Η κεντρική τράπεζα διατήρησε το βασικ...,Οι επιπτώσεις από την κρίση στη Μέση Ανατολή.,https://www.capital.gr/diethni/3988002/iaponia...


In [36]:
pd.DataFrame(data)

,Τίτλος,Περίληψη,Link
0,US political violence generates a familiar cyc...,"In modern America, it seems violence of this k...",https://www.bbc.com/news/articles/cje4eeyq27lo
1,United Arab Emirates to quit oil cartel Opec,"The UAE's decision, after nearly 60 years of m...",https://www.bbc.com/news/articles/cj4pxwlr52yo
2,Why Sam Altman and his former hero Elon Musk a...,The battle between the AI big hitters has larg...,https://www.bbc.com/news/articles/cn8dedv8w8xo
3,'It's bizarre': Californians grapple with reve...,People living near the man charged with the at...,https://www.bbc.com/news/articles/c392knk3w2jo
4,Watch: Jimmy Kimmel defends 'expectant widow' ...,First Lady Melania Trump described the joke as...,https://www.bbc.com/news/videos/c62d4n5rzr7o
5,Russian superyacht sails through Strait of Hor...,"The 141m-long vessel, linked to a close ally o...",https://www.bbc.com/news/articles/cm2pn8zdxdjo
6,The other life of US soldier accused of bettin...,The Army officer charged with using classified...,https://www.bbc.com/news/articles/clyxd5wrr0wo
7,Suspected gunman aged 89 held in Greece after ...,Δεν υπάρχει,https://www.bbc.com/news/articles/cyv2pmz2v86o
8,Canada's Carney launches a sovereign wealth fu...,Δεν υπάρχει,https://www.bbc.com/news/articles/c98m800r28qo
9,Ex-actor Nathan Chasing Horse jailed for at le...,Δεν υπάρχει,https://www.bbc.com/news/articles/cx218vkdvpgo


## ΒΗΜΑ ΤΕΣΣΕΡΑ: Χρησιμοποιούμε την είδηση για να πάρουμε την σύνοψη

Το ίδιο και πάλι! Αυτή τη φορά ψάχνουμε τις παραγράφους `p`.

In [ ]:
stories = doc.find_all('div', { 'class': 'gs-c-promo' })
for story in stories:
    print("THIS IS A STORY")
    headline = story.find('h3')
    print(headline.text)
    link = story.find('a')
    print(link['href'])
    summary = story.find('p')
    print(summary.text)

### Λείπουν elements

Αυτό μας έλειπε τώρα, ένα error! Συγκεκριμένα λέει ότι ...

    ---------------------------------------------------------------------------
    AttributeError                            Traceback (most recent call last)
    <ipython-input-20-e55795264040> in <module>()
          7     print(link['href'])
          8     summary = story.find('p')
    ----> 9     print(summary.text)

    AttributeError: 'NoneType' object has no attribute 'text'

Εφόσον το λάθος εμφανίστηκε από την στιγμή που βάλαμε το κομμάτι με το `summary` part, υποθέτουμε ότι το πρόβλημα είναι ότι δεν έχει κάθε μια **ιστορία από ένα summary**. Πώς θα το λύσουμε αυτό???!!!

Λοιπόν, μπορούμε να *ρωτάμε αν (if) η ιστορία έχει summary*. Αν έχει τότε μπορούμε να το χρησιμοποιήσουμε. Αν δεν έχει το αγνοούμε. **Είναι ένα απλό `if` statement**.

In [ ]:
stories = doc.find_all('div', { 'class': 'gs-c-promo' })
for story in stories:
    print("THIS IS A STORY")
    headline = story.find('h3')
    print(headline.text)
    link = story.find('a')
    print(link['href'])
    summary = story.find('p')
    if summary:
        print(summary.text)

## Ας το σώσουμε ως CSV

Τώρα που έχουμε όλα τα στοιχεία που θέλαμε μπορούμε να τα μετατρέψουμε σε ένα CSV. Υπάρχουν 3 βήματα για τη μετατροπή σε CSV:
    
1. **Ξεκινάμε με μια άδεια λίστα:** Κάθε ιστορία που βρίσκουμε, την προσθέτουμε στη λίστα
2. **Φτιάχνουμε ένα λεξικό** για κάθε μια ιστορία
3. **Μετατρέπουμε τη λίστα σε ένα DataFrame**, μετά
4. **Εξάγουμε το DataFrame σε ένα CSV**

Η δημιουργία του λεξικού μπορεί να είναι λίγο περίπλοκη, οπότε ας δούμε **2 διαφορετικούς τρόπους να το κάνουμε**.

### Μέθοδος 1η: Όλα μαζί

Θα φτιάξουμε το λεξικό `story_dict` απευθείας και μετά θα το προσθέσουμε στη λίστα `stories_list`.

In [ ]:
# Ξεκινάμε με μια άδεια λίστα
stories_list = []
stories = doc.find_all('div', { 'class': 'gs-c-promo' })
for story in stories:
    headline = story.find('h3')
    link = story.find('a')
    summary = story.find('p')
    # Έχει η ιστορία summary?
    if summary:
        summary_text = summary.text
    else:
        summary_text = ''

        # Φτιάξε ένα λεξικό αν ΕΧΕΙ summary
    story_dict = {
            'headline': headline.text,
            'url': link['href'],
            'summary': summary_text
        }

    # Πρόσθεσε το λέξικο στη λίστα
    stories_list.append(story_dict)

print(stories_list)

# Τώρα που τελειώσαμε μετέτρεψε το σε CSV και αποθήκευσε το.
# Αν δεν χρησιμοποιήσετε το index=False, θα έχετε ένα άσχημο dataframe!
import pandas as pd
df = pd.DataFrame(stories_list)
df.to_csv("bbc.csv", index=False)

### Μέθοδος 2η: Γεμίζοντας τα κενά

Για αυτή τη μέθοδο θα φτιάξουμε πρώτα το λεξικό μας `story_dict` και μετά θα το γεμίζουμε με τα στοιχεία που υπάρχουν.

In [ ]:
# Ξεκινάμε με μια άδεια λίστα
stories_list = []
stories = doc.find_all('div', { 'class': 'gs-c-promo' })
for story in stories:
    # Φτιάχνουμε ένα λεξικό χωρίς τίποτα
    story_dict = {}
    headline = story.find('h3')
    if headline:
        story_dict['headline'] = headline.text
    link = story.find('a')
    if link:
        story_dict['url'] = link['href']
    summary = story.find('p')
    if summary:
        story_dict['summary'] = summary.text
    # Προσθέτουμε το λεξικό στη λίστα
    stories_list.append(story_dict)

# Τώρα που τελειώσαμε μετέτρεψε το σε CSV και αποθήκευσε το.
# Αν δεν χρησιμοποιήσετε το index=False, θα έχετε ένα άσχημο dataframe!
import pandas as pd
df = pd.DataFrame(stories_list)
df.to_csv("bbc.csv", index=False)